# Explicabilité (Module 5) : Telco Churn

Ce notebook utilise exactement les fonctions que l'application et l'API appellent (`src/explain_service.py`).

1. Entraîner les modèles  2. Explications globales  3. Explication d'un client  4. Équité  5. Analyse critique

> Lancer depuis la racine du projet avec le venv actif. Compter 2 à 5 minutes (entraînement avec optimisation).

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.append(os.getcwd())

import numpy as np
import pandas as pd

from src import data as D, trainer as T, models as M, explain_service as EX, xai_plots as XP

## 1. Entraînement (même code que l'API)

In [2]:
df = D.generic_clean(D.load_data())
df["Churn"] = (df["Churn"] == "Yes").astype(int)

session = T.run_training({"target": "Churn", "tune": True, "n_iter": 12, "cv": 3}, df)
results = M.evaluate_scores(session["scores"], session["y_test"], 0.5)
results[["AUC-ROC", "PR-AUC", "Précision", "Rappel", "F1"]]

,AUC-ROC,PR-AUC,Précision,Rappel,F1
XGBoost,0.8459,0.6584,0.5128,0.8048,0.6264
Random Forest,0.8452,0.6572,0.5219,0.7968,0.6307
Logistic Regression,0.8409,0.6273,0.5052,0.7807,0.6134
Decision Tree,0.8361,0.6220,0.5244,0.8048,0.6350
KNN,0.8304,0.6226,0.6068,0.5695,0.5876


In [3]:
model_name = results.index[0]          # le meilleur modèle
print("Modèle expliqué :", model_name)

Modèle expliqué : XGBoost


## 2. Explications globales : le modèle dans son ensemble

**Importance propre au modèle** : coefficients (régression logistique) ou importance par impureté (arbres).

In [4]:
nat = EX.native(session, "Logistic Regression")
XP.plot_native_importance(nat["importance"]).show()

**Importance par permutation** : baisse d'AUC quand on mélange une variable. Elle se compare entre modèles.

In [5]:
perm = EX.permutation(session, n_repeats=5)
XP.plot_permutation_comparison(perm["importance"]).show()

**SHAP summary plot** : chaque point est un client. À droite, la variable augmente le risque ; en rouge, sa valeur est élevée.

In [6]:
sg = EX.shap_global(session, model_name, n=300)
XP.plot_beeswarm(sg["values"], sg["data"], sg["feature_names"]).show()

In [7]:
imp = pd.DataFrame(sg["importance_by_variable"])
imp.head(10)

,Variable,Importance SHAP moyenne,Part
0,Contract,0.788107,0.303919
1,InternetService,0.403339,0.155540
2,tenure,0.378040,0.145784
3,PaymentMethod,0.187426,0.072278
4,MonthlyCharges,0.150716,0.058121
5,PaperlessBilling,0.123107,0.047474
6,has_protection,0.120206,0.046355
7,StreamingMovies,0.104304,0.040223
8,TotalCharges,0.074768,0.028833
9,OnlineSecurity,0.048055,0.018532


In [8]:
# Dépendance SHAP : effet d'une variable selon sa valeur
for f in ["tenure", "MonthlyCharges"]:
    XP.plot_dependence(sg["values"], sg["data"], sg["feature_names"], f).show()

## 3. Explication locale : un client donné

In [9]:
# Un client du jeu de test réellement parti
i = int(np.where(session["y_test"] == 1)[0][0])
ind = EX.get_individual(session, i)
print("Réalité :", "parti" if ind["true_label"] else "resté")
ind["data"]

Réalité : parti


{'gender': 'Male',
 'SeniorCitizen': 1,
 'Partner': 'No',
 'Dependents': 'No',
 'tenure': 17,
 'PhoneService': 'Yes',
 'MultipleLines': 'No',
 'InternetService': 'DSL',
 'OnlineSecurity': 'No',
 'OnlineBackup': 'No',
 'DeviceProtection': 'No',
 'TechSupport': 'No',
 'StreamingTV': 'No',
 'StreamingMovies': 'No',
 'Contract': 'Month-to-month',
 'PaperlessBilling': 'Yes',
 'PaymentMethod': 'Electronic check',
 'MonthlyCharges': 45.05,
 'TotalCharges': 770.6}

In [10]:
res = EX.compare_local(session, model_name, ind["data"], threshold=0.5)
print(f"Probabilité de départ : {res['probability']:.1%}")
print(res["summary"])

Probabilité de départ : 60.7%
Le risque de départ est élevé (61%, seuil de décision 50%), principalement à cause de : contrat = Month-to-month, mode de paiement = Electronic check et protection (sécurité ou support) = 0. Il est atténué par : service internet = DSL.


In [11]:
sh = res["shap"]
contrib = pd.DataFrame(sh["contributions"])
contrib = pd.concat([contrib, pd.DataFrame([{"label": f"{sh['n_other_features']} autres variables",
                                             "contribution": sh["other_contribution"]}])], ignore_index=True)
XP.plot_waterfall(contrib, sh["base_value"], sh["units"], res["probability"]).show()

In [12]:
XP.plot_lime(res["lime"]["rules"]).show()
print("R² de l'approximation locale de LIME :", round(res["lime"]["local_fit_r2"], 2))

R² de l'approximation locale de LIME : 0.68


In [13]:
cm = res["comparison"]
print(f"Variables communes SHAP/LIME dans le top {cm['k']} : {cm['overlap']}/{cm['k']}")
print(f"Accord sur le sens de l'effet : {cm['sign_agreement']:.0%}")
pd.DataFrame(cm["table"])[["label", "SHAP", "LIME", "Rang SHAP", "Rang LIME", "Même sens"]]

Variables communes SHAP/LIME dans le top 5 : 4/5
Accord sur le sens de l'effet : 100%


,label,SHAP,LIME,Rang SHAP,Rang LIME,Même sens
0,contrat = Month-to-month,0.528313,0.332132,1.0,1.0,True
1,service internet = DSL,-0.316963,-0.030738,2.0,NaN,True
2,mode de paiement = Electronic check,0.280321,0.083742,3.0,2.0,True
3,protection (sécurité ou support) = 0,0.098069,0.059325,4.0,3.0,True
4,facture dématérialisée = Yes,0.090878,0.058460,5.0,4.0,True
5,films en streaming = No,-0.067271,-0.046509,NaN,5.0,True


## 4. Variables sensibles et équité

In [14]:
sens = ["gender", "SeniorCitizen", "Partner", "Dependents"]
fr_ = EX.fairness(session, model_name, threshold=0.5, sensitive=sens)
rep = pd.DataFrame(fr_["report"])
display(rep.round(3))
display(pd.DataFrame(fr_["gaps"]).round(3))

part = imp[imp["Variable"].isin(sens)]["Part"].sum()
print(f"Part de l'importance SHAP portée par les variables sensibles : {part:.1%}")

,Variable,Groupe,Effectif,Taux réel positif,Taux prédit positif,Proba moyenne,Rappel,Faux positifs (FPR)
0,gender,Male,722,0.251,0.424,0.413,0.829,0.288
1,gender,Female,687,0.281,0.409,0.413,0.782,0.263
2,SeniorCitizen,0,1187,0.233,0.369,0.382,0.768,0.248
3,SeniorCitizen,1,222,0.441,0.671,0.581,0.908,0.484
4,Partner,Yes,673,0.187,0.284,0.322,0.706,0.186
5,Partner,No,736,0.337,0.538,0.497,0.855,0.377
6,Dependents,Yes,431,0.165,0.225,0.275,0.648,0.142
7,Dependents,No,978,0.310,0.501,0.474,0.842,0.348


,Variable,Écart de parité
0,Dependents,0.276
1,Partner,0.254
2,SeniorCitizen,0.302
3,gender,0.015


Part de l'importance SHAP portée par les variables sensibles : 2.4%


## 5. Analyse critique (à rédiger pour le one-pager)

**Cohérence SHAP / LIME / intuition métier**
- SHAP et LIME s'accordent-ils (variables communes, sens des effets) sur plusieurs clients ? Désaccords et causes possibles (variables corrélées, instabilité de LIME) :
- Les effets (contrat mensuel, ancienneté, fibre, support technique) sont-ils conformes à l'intuition métier ? :

**Variables sensibles et équité**
- Poids de `gender`, `SeniorCitizen`, `Partner`, `Dependents` dans le modèle (part SHAP) :
- Écarts de taux de prédictions positives entre groupes (parité) :
- Risque de variables substituts (facture élevée, mode de paiement) :

**Performance vs interprétabilité**
- AUC du meilleur modèle vs régression logistique (écart) :
- Recommandation au décideur :

**Actions concrètes**
- Contrats d'un an ou plus pour les clients à risque, ciblage des nouveaux clients, promotion du support technique et de la sécurité, revue de l'offre fibre :

**Limites**
- Association ≠ causalité ; un seul jeu de données et un seul découpage ; SHAP et LIME sont des approximations ; les explications locales de LIME peuvent varier d'un tirage à l'autre.